# 🔍 SSIM Autoencoder for MVTec Anomaly Detection — Bottle Class

This notebook trains an **SSIM-based Convolutional Autoencoder** for unsupervised anomaly detection on the **MVTec AD** dataset (Bottle class).

**Key Ideas:**
- Train only on **normal** images (good class)
- At inference, anomalies produce high **SSIM reconstruction error**
- Anomaly map = pixel-wise SSIM difference

**Reference:** *Improving Unsupervised Defect Segmentation by Applying Structural Similarity to Autoencoders* (Bergmann et al., 2018)

In [ ]:
# ============================================================
# 1. INSTALL & IMPORTS
# ============================================================
!pip install pytorch-msssim -q

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from pytorch_msssim import ssim, SSIM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# 2. CONFIGURATION  ← Set your MVTec bottle path here
# ============================================================

# Example: '/content/drive/MyDrive/mvtec_anomaly_detection/bottle'
DATASET_PATH = '/content/bottle'   # <-- CHANGE THIS

IMG_SIZE     = 256      # Resize images to this square size
BATCH_SIZE   = 16
EPOCHS       = 100
LR           = 2e-4
LATENT_DIM   = 512      # bottleneck channels
SSIM_WINDOW  = 11       # SSIM kernel size
SAVE_PATH    = 'ssim_autoencoder_bottle.pth'

# Expected MVTec structure:
# bottle/
#   train/good/*.png
#   test/good/*.png
#   test/broken_large/*.png
#   test/broken_small/*.png
#   test/contamination/*.png
#   ground_truth/ ...
print('Config OK')

In [ ]:
# ============================================================
# 3. DATASET
# ============================================================

class MVTecDataset(Dataset):
    """Loads MVTec images. For training: only 'good' class.
       For testing: all subfolders with labels (good=0, anomaly=1)."""

    def __init__(self, root, split='train', img_size=256):
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),          # [0,1]
        ])
        self.images = []
        self.labels = []  # 0=normal, 1=anomaly

        split_dir = os.path.join(root, split)
        assert os.path.isdir(split_dir), f'Split directory not found: {split_dir}'

        for cls_name in sorted(os.listdir(split_dir)):
            cls_dir = os.path.join(split_dir, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            label = 0 if cls_name == 'good' else 1
            for ext in ('*.png', '*.jpg', '*.bmp'):
                for fp in sorted(glob.glob(os.path.join(cls_dir, ext))):
                    self.images.append(fp)
                    self.labels.append(label)

        print(f'[{split}] {len(self.images)} images | '
              f'normal={self.labels.count(0)}, anomaly={self.labels.count(1)}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]


train_dataset = MVTecDataset(DATASET_PATH, split='train', img_size=IMG_SIZE)
test_dataset  = MVTecDataset(DATASET_PATH, split='test',  img_size=IMG_SIZE)

# Training uses only 'good' samples
good_indices = [i for i, l in enumerate(train_dataset.labels) if l == 0]
train_subset = torch.utils.data.Subset(train_dataset, good_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=1,          shuffle=False, num_workers=2, pin_memory=True)

print(f'Training on {len(train_subset)} normal samples')

In [ ]:
# ============================================================
# 4. SSIM AUTOENCODER MODEL
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True)
        )
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, activation=True):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
        ]
        if activation:
            layers += [nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class SSIMAutoencoder(nn.Module):
    """Encoder compresses 256x256 → 4x4 latent. Decoder reconstructs back."""

    def __init__(self, in_channels=3, latent_channels=512):
        super().__init__()

        # Encoder: 256 → 128 → 64 → 32 → 16 → 8 → 4
        self.encoder = nn.Sequential(
            ConvBlock(in_channels, 32,  stride=2),  # 128
            ConvBlock(32,  64,  stride=2),           # 64
            ConvBlock(64,  128, stride=2),           # 32
            ConvBlock(128, 256, stride=2),           # 16
            ConvBlock(256, latent_channels, stride=2),  # 8
            ConvBlock(latent_channels, latent_channels, stride=2),  # 4
        )

        # Decoder: 4 → 8 → 16 → 32 → 64 → 128 → 256
        self.decoder = nn.Sequential(
            DeconvBlock(latent_channels, latent_channels),  # 8
            DeconvBlock(latent_channels, 256),              # 16
            DeconvBlock(256, 128),                          # 32
            DeconvBlock(128, 64),                           # 64
            DeconvBlock(64, 32),                            # 128
            DeconvBlock(32, in_channels, activation=False), # 256
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


model = SSIMAutoencoder(in_channels=3, latent_channels=LATENT_DIM).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

# Quick shape test
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
    out, z = model(dummy)
    print(f'Input: {dummy.shape} → Latent: {z.shape} → Output: {out.shape}')

In [ ]:
# ============================================================
# 5. SSIM LOSS FUNCTION
# ============================================================

class SSIMLoss(nn.Module):
    """Combined SSIM + L1 loss for better texture reconstruction."""
    def __init__(self, window_size=11, alpha=0.84):
        super().__init__()
        self.ssim_fn = SSIM(win_size=window_size, data_range=1.0, size_average=True, channel=3)
        self.alpha = alpha  # weight for SSIM component

    def forward(self, pred, target):
        ssim_loss = 1 - self.ssim_fn(pred, target)
        l1_loss   = nn.functional.l1_loss(pred, target)
        return self.alpha * ssim_loss + (1 - self.alpha) * l1_loss


criterion = SSIMLoss(window_size=SSIM_WINDOW, alpha=0.84)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print('Loss function and optimizer ready')

In [ ]:
# ============================================================
# 6. TRAINING LOOP
# ============================================================

train_losses = []
best_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for imgs, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        recon, _ = model(imgs)
        loss = criterion(recon, imgs)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), SAVE_PATH)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d}/{EPOCHS} | Loss: {avg_loss:.5f} | LR: {scheduler.get_last_lr()[0]:.2e}')

print(f'\nTraining complete. Best loss: {best_loss:.5f}')

In [ ]:
# ============================================================
# 7. PLOT TRAINING CURVE
# ============================================================

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='SSIM+L1 Loss', color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss — SSIM Autoencoder (Bottle)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 8. ANOMALY SCORING (pixel-wise SSIM residual map)
# ============================================================

def compute_anomaly_score(model, img_tensor):
    """Returns scalar anomaly score and pixel-level heatmap."""
    model.eval()
    with torch.no_grad():
        img = img_tensor.unsqueeze(0).to(device)  # [1,C,H,W]
        recon, _ = model(img)
        # Per-pixel SSIM (returns map in [0,1], higher=more similar)
        ssim_map = ssim(img, recon, data_range=1.0, size_average=False,
                        win_size=SSIM_WINDOW)
        # Anomaly map: 1 - SSIM (higher = more anomalous)
        anomaly_map = (1 - ssim_map).squeeze().cpu().numpy()  # [H,W]
        score = anomaly_map.mean()
    return score, anomaly_map


# Load best checkpoint
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))

all_scores, all_labels = [], []
for imgs, labels in tqdm(test_loader, desc='Evaluating'):
    score, _ = compute_anomaly_score(model, imgs[0])
    all_scores.append(score)
    all_labels.append(labels.item())

auc = roc_auc_score(all_labels, all_scores)
print(f'\n✅ Image-level AUROC: {auc:.4f}')

In [ ]:
# ============================================================
# 9. VISUALIZE RECONSTRUCTIONS & ANOMALY MAPS
# ============================================================

def show_results(model, dataset, n=4, title=''):
    model.eval()
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    indices = list(range(min(n, len(dataset))))
    for row, idx in enumerate(indices):
        img, label = dataset[idx]
        score, amap = compute_anomaly_score(model, img)

        orig_np = img.permute(1, 2, 0).numpy()
        with torch.no_grad():
            recon = model(img.unsqueeze(0).to(device))[0].squeeze().cpu().permute(1, 2, 0).numpy()

        axes[row, 0].imshow(np.clip(orig_np, 0, 1))
        axes[row, 0].set_title(f'Input  | label={"anomaly" if label else "good"}')

        axes[row, 1].imshow(np.clip(recon, 0, 1))
        axes[row, 1].set_title('Reconstruction')

        im = axes[row, 2].imshow(amap, cmap='hot', vmin=0, vmax=0.5)
        axes[row, 2].set_title(f'Anomaly Map | score={score:.4f}')
        plt.colorbar(im, ax=axes[row, 2])

        for ax in axes[row]: ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=150)
    plt.show()


# Show normal test samples
normal_test = torch.utils.data.Subset(test_dataset, [i for i, l in enumerate(test_dataset.labels) if l == 0][:4])
show_results(model, normal_test, n=4, title='Normal Samples')

# Show anomaly test samples
anomaly_test = torch.utils.data.Subset(test_dataset, [i for i, l in enumerate(test_dataset.labels) if l == 1][:4])
show_results(model, anomaly_test, n=4, title='Anomaly Samples')

In [ ]:
# ============================================================
# 10. THRESHOLD SELECTION & PIXEL-LEVEL EVALUATION
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
roc_auc = auc(fpr, tpr)

# Youden's J statistic for optimal threshold
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f'Optimal threshold (Youden J): {optimal_threshold:.6f}')
print(f'At threshold: TPR={tpr[optimal_idx]:.4f}, FPR={fpr[optimal_idx]:.4f}')

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', label=f'ROC Curve (AUC={roc_auc:.4f})')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', s=100,
            label=f'Optimal threshold={optimal_threshold:.4f}', zorder=5)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — SSIM Autoencoder (MVTec Bottle)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()

## Summary

| Step | Description |
|------|-------------|
| **Model** | Convolutional Autoencoder (256→4 latent, 4→256 decode) |
| **Loss** | SSIM (α=0.84) + L1 (α=0.16) |
| **Training** | Only on normal (good) images |
| **Scoring** | Mean(1 - SSIM(input, reconstruction)) |
| **Evaluation** | Image-level AUROC |

**Tips for better performance:**
- Increase epochs (200-300) for better convergence
- Add random augmentation (flips, brightness jitter) to training
- Try patch-based SSIM scoring for finer anomaly localization
- Use a larger latent dim if underfitting